# Gravitino MCP: metadata for agents

This notebook connects to the **Gravitino MCP server** running in the playground and drives it three ways, from low-level to fully agentic:

1. **Connect & discover** — open an MCP session and list the tools Gravitino exposes to an agent.
2. **Govern at the source** — walk a real table and inspect the tags and policies attached to it. No LLM required.
3. **Ask in natural language** — hand the tools to an LLM and let it answer a question by calling them itself. Requires an API key.

The point: an agent connected here does not get raw files. It gets *governed* metadata — catalogs, schemas, tables, tags, policies — through a uniform tool surface, with every call subject to Gravitino's authorization and captured in its audit log.

> The MCP server is reachable from inside the playground network at `http://gravitino-mcp:8000/mcp`. From your laptop it is `http://localhost:8000/mcp` (directly, or through an SSH tunnel to the host).

## 0. Install the client libraries

Self-contained, following the playground convention of installing per-notebook. `mcp` is the official Model Context Protocol SDK; the LlamaIndex packages are only needed for the agentic section at the end.

In [ ]:
%pip install -q "mcp>=1.2.0"
# Only needed for the agentic section (Section 3). Both provider LLMs are
# installed so the notebook works with whichever key you have (Anthropic or OpenAI).
%pip install -q "llama-index-core" "llama-index-tools-mcp" \
    "llama-index-llms-anthropic" "llama-index-llms-openai"

## 1. Connect & discover

Open a streamable-HTTP MCP session, run the `initialize` handshake, and list the available tools. Jupyter supports top-level `await`, so we can call the async client directly.

In [ ]:
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

# In-network address of the MCP server (compose service name). If you run this
# notebook outside the playground, change to http://localhost:8000/mcp.
MCP_URL = "http://gravitino-mcp:8000/mcp"

async def list_tools():
    async with streamablehttp_client(MCP_URL) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            resp = await session.list_tools()
            return resp.tools

tools = await list_tools()
print(f"Gravitino exposes {len(tools)} MCP tools:\n")
for t in tools:
    print(f"  {t.name}")

Notice the shape of that surface. Alongside plain metadata navigation (`get_list_of_catalogs`, `get_list_of_tables`) there are **tag** tools, **policy** tools, and **job** tools. That is the difference between a raw table format and a governed semantic layer: an agent here can ask not just *what tables exist* but *what is sensitive*, *what rules apply*, and *what work can run*.

## 2. Govern at the source (no LLM)

A small helper to call any tool, then a scripted traversal: list catalogs, drill into a schema, pick a table, and inspect the governance metadata attached to it.

In [ ]:
import json

async def call_tool(name, args=None):
    async with streamablehttp_client(MCP_URL) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool(name, args or {})
            # Tool results arrive as content blocks; pull the text payloads.
            texts = [b.text for b in result.content if getattr(b, "type", None) == "text"]
            return "\n".join(texts)

catalogs = await call_tool("get_list_of_catalogs")
print(catalogs)

In [ ]:
# Drill into the Iceberg catalog's analytics schema. The 'orders' table is
# seeded by the playground's Iceberg demo data.
tables = await call_tool("get_list_of_tables", {
    "catalog_name": "catalog_iceberg",
    "schema_name": "analytics",
})
print(tables)

In [ ]:
# Full metadata for a specific table.
details = await call_tool("get_table_metadata_details", {
    "catalog_name": "catalog_iceberg",
    "schema_name": "analytics",
    "table_name": "orders",
})
print(details)

### Seed some governance metadata

The playground ships without tags or policies, so the tools above return empty lists on a fresh start. Here we create two tags and attach them to the `orders` table so the governance surface has something real to show. Tag *creation* is a Gravitino REST call (there is no create-tag MCP tool by design); *association* can be done via REST or the `associate_tag_with_metadata` MCP tool. This cell is idempotent, re-running it is harmless.

In [ ]:
import requests

GRAVITINO = "http://gravitino:8090"  # in-network address of the Gravitino server
METALAKE = "metalake_demo"
HDRS = {"Accept": "application/vnd.gravitino.v1+json", "Content-Type": "application/json"}

def create_tag(name, comment):
    r = requests.post(
        f"{GRAVITINO}/api/metalakes/{METALAKE}/tags",
        json={"name": name, "comment": comment, "properties": {}},
        headers=HDRS,
    )
    if r.status_code == 200:
        print(f"tag {name}: created")
    elif r.status_code == 409:
        print(f"tag {name}: already exists (ok)")
    else:
        print(f"tag {name}: unexpected {r.status_code} {r.text[:120]}")

def tag_table(catalog, schema, table, tags):
    obj = f"{catalog}.{schema}.{table}"
    r = requests.post(
        f"{GRAVITINO}/api/metalakes/{METALAKE}/tags/table/{obj}",
        json={"tagsToAdd": tags},
        headers=HDRS,
    )
    if r.status_code == 200:
        print(f"{obj}: tagged with {tags}")
    elif r.status_code == 409:
        print(f"{obj}: already tagged with {tags} (ok)")
    else:
        print(f"{obj}: unexpected {r.status_code} {r.text[:120]}")

create_tag("PII", "Contains personally identifiable information")
create_tag("sensitive", "Business-sensitive data, restricted access")
tag_table("catalog_iceberg", "analytics", "orders", ["PII", "sensitive"])

## The governance payoff

Now the part a raw catalog cannot give an agent. List the tags in the metalake, and the policies, then ask which are attached to our table. In a fresh playground these may be empty until you create some (see the tag and access-control examples), but the *tools are live* — this is the surface an agent uses to reason about sensitivity and rules.

In [ ]:
print("--- tags in the metalake ---")
print(await call_tool("list_of_tags"))
print("\n--- objects carrying the PII tag ---")
# Reverse direction (tag -> objects) is the reliable listing path and shows
# our tagged 'orders' table under the PII tag.
print(await call_tool("list_metadata_by_tag", {"tag_name": "PII"}))
print("\n--- policies in the metalake ---")
print(await call_tool("get_list_of_policies"))

> **Identity & authorization.** Every call above carried whatever identity the MCP session presented. In the default playground that is `anonymous`; with authentication enabled, the MCP server forwards your `Authorization` header to Gravitino, which authorizes each call per principal and records it in `gravitino-mcp-audit.log`. Same tools, different answers depending on who is asking — governance enforced at the source, not bolted onto the agent.

## 3. Ask in natural language (agentic)

This is the beat that matters for the "why agents need a governed catalog" story. We wrap the MCP tools as agent tools and let an LLM decide which to call to answer a plain-English question. It runs a real tool-use loop: the model reads the tool list, calls `get_list_of_tables` and friends on its own, and composes an answer from what it finds.

Set an API key to enable it. If no key is set in the environment, the next cell prompts you for one (masked, kept only in this session). Leave the prompt blank to skip the agent; everything above still stands on its own.

In [ ]:
import os
# Use whichever provider key is available. Anthropic is preferred if both are set.
# The two providers are not interchangeable by key alone, so we pick the SDK that
# matches the key you actually have.
PROVIDER = None
if os.environ.get("ANTHROPIC_API_KEY"):
    PROVIDER = "anthropic"
elif os.environ.get("OPENAI_API_KEY"):
    PROVIDER = "openai"
else:
    try:
        import getpass
        entered = getpass.getpass("No provider key set. Paste an ANTHROPIC or OPENAI key (or leave blank to skip): ")
        if entered.startswith("sk-ant"):
            os.environ["ANTHROPIC_API_KEY"] = entered; PROVIDER = "anthropic"
        elif entered:
            os.environ["OPENAI_API_KEY"] = entered; PROVIDER = "openai"
    except Exception:
        pass

AGENT_ENABLED = PROVIDER is not None
print(f"Agent enabled using {PROVIDER}" if AGENT_ENABLED else "No key set — skipping the agentic section.")

In [ ]:
if AGENT_ENABLED:
    from llama_index.tools.mcp import BasicMCPClient, McpToolSpec
    from llama_index.core.agent.workflow import FunctionAgent

    mcp_client = BasicMCPClient(MCP_URL)
    tool_spec = McpToolSpec(client=mcp_client)
    agent_tools = await tool_spec.to_tool_list_async()

    if PROVIDER == "anthropic":
        from llama_index.llms.anthropic import Anthropic
        # Set ANTHROPIC_MODEL to a current id from https://docs.claude.com/en/docs/about-claude/models
        llm = Anthropic(model=os.environ.get("ANTHROPIC_MODEL", "claude-sonnet-4-5"))
    else:
        from llama_index.llms.openai import OpenAI
        llm = OpenAI(model=os.environ.get("OPENAI_MODEL", "gpt-4o"))

    agent = FunctionAgent(
        tools=agent_tools,
        llm=llm,
        system_prompt=(
            "You are a data governance assistant. Use the Gravitino MCP tools to "
            "answer questions about catalogs, schemas, tables, tags, and policies. "
            "Explain what you find in plain language."
        ),
    )
    print(f"Agent ready with {len(agent_tools)} Gravitino tools (provider: {PROVIDER}).")

In [ ]:
if AGENT_ENABLED:
    from IPython.display import Markdown, display
    response = await agent.run(
        "What catalogs are available, and what tables are in the analytics "
        "schema of the Iceberg catalog? For each table, note any tags or "
        "policies attached to it."
    )
    # The agent replies in Markdown; render it rather than printing raw text.
    display(Markdown(str(response)))

That question never named a tool or an endpoint. The model discovered the catalogs, drilled into the schema, and checked governance metadata by calling the MCP tools itself, then answered in prose. That is the experience a customer's agent gets against their own Gravitino: natural-language questions, answered from live, governed metadata, with authorization and audit intact underneath.

### Where to take it next
- Turn on authentication (`./playground.sh start --enable-auth`) and pass an `Authorization` header, then watch the same question return different results per principal.
- Create tags and policies (see the tag and access-control notebooks) so the governance answers become non-empty.
- Point the agent at `run_job` to have it trigger governed maintenance work, not just read metadata.